Step 1: Load and prepare documents

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# List of URLs to load documents from
urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

# Load documents from the URLs
docs = [WebBaseLoader(
                url, 
                requests_kwargs={"headers": {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}}
            ).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

Step 2: Split documents into chunks

In [ ]:
# Initialize a text splitter with specified chunk size and overlap
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250, chunk_overlap=0
)
# Split the documents into chunks
doc_splits = text_splitter.split_documents(docs_list)

Step 3: Create a vector store

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_openai import AzureOpenAIEmbeddings

# Azure OpenAI Configuration  
azure_openai_endpoint = "https://swedencentral.api.cognitive.microsoft.com/"
azure_openai_api_key = "YOUR_API_KEY_HERE"  # Get from Azure Portal
azure_openai_api_version = "2024-02-15-preview"
embedding_deployment = "text-embedding-ada-002"

# Create embeddings and store document chunks in ChromaDB
vectorstore = Chroma.from_documents(
    documents=doc_splits,
    embedding=AzureOpenAIEmbeddings(
        azure_endpoint=azure_openai_endpoint,
        azure_deployment=embedding_deployment,
        openai_api_key=azure_openai_api_key,
        openai_api_version=azure_openai_api_version,
    ),
    collection_name="rag_docs",
    persist_directory="./chroma_db",
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

In [ ]:
from langchain_community.vectorstores import SKLearnVectorStore  
from langchain_openai import AzureOpenAIEmbeddings

# Azure OpenAI Configuration  
azure_openai_endpoint = "https://swedencentral.api.cognitive.microsoft.com/"
azure_openai_api_key = "YOUR_API_KEY_HERE"  # Get from Azure Portal
azure_openai_api_version = "2024-02-15-preview"
embedding_deployment = "text-embedding-ada-002"

# Create embeddings for documents and store them in a vector store
vectorstore = SKLearnVectorStore.from_documents(
    documents=doc_splits,
    embedding=AzureOpenAIEmbeddings(
        azure_endpoint=azure_openai_endpoint,
        azure_deployment=embedding_deployment,
        openai_api_key=azure_openai_api_key,
        openai_api_version=azure_openai_api_version,
    ),
)
retriever = vectorstore.as_retriever(k=4)

Step 4: Set up the LLM and prompt template

In [ ]:
from langchain_openai import AzureChatOpenAI
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Define the prompt template for the LLM
prompt = PromptTemplate(
    template="""You are an assistant for question-answering tasks.
    Use the following documents to answer the question.
    If you don't know the answer, just say that you don't know.
    Use three sentences maximum and keep the answer concise:
    Question: {question}
    Documents: {documents}
    Answer:
    """,
    input_variables=["question", "documents"],
)

# Initialize the Azure OpenAI LLM
llm = AzureChatOpenAI(
    azure_endpoint=azure_openai_endpoint,
    azure_deployment="gpt-4o-mini",
    openai_api_key=azure_openai_api_key,
    openai_api_version=azure_openai_api_version,
    temperature=0,
)

# Create a chain combining the prompt template and LLM
rag_chain = prompt | llm | StrOutputParser()

Step 5: Integrate the retriever and LLM into a RAG application

In [ ]:
# Define the RAG application class
class RAGApplication:
    def __init__(self, retriever, rag_chain):
        self.retriever = retriever
        self.rag_chain = rag_chain
    def run(self, question):
        # Retrieve relevant documents
        documents = self.retriever.invoke(question)
        # Extract content from retrieved documents
        doc_texts = "\\n".join([doc.page_content for doc in documents])
        # Get the answer from the language model
        answer = self.rag_chain.invoke({"question": question, "documents": doc_texts})
        return answer

Step 6: Test your RAG application

In [ ]:
# Create and test the RAG application
rag_app = RAGApplication(retriever, rag_chain)

# Test with some questions
questions = [
    "What are the key components of an AI agent?",
    "What is prompt engineering?",
    "How do adversarial attacks work on large language models?"
]

# Test each question
for question in questions:
    print(f"\n🤖 Question: {question}")
    print("-" * 50)
    try:
        answer = rag_app.run(question)
        print(f"📋 Answer: {answer}")
    except Exception as e:
        print(f"❌ Error: {e}")
    print("=" * 50)